In [1]:
import numpy as np
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
env_name = "FrozenLake-v1"
env = gym.make(env_name, is_slippery=False, new_step_api=True)

In [9]:
print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)
policy = np.random.choice(env.action_space.n, size=env.observation_space.n)
value_table = np.zeros(env.observation_space.n)
gamma = 0.9
theta = 1e-10
def policy_evaluation(policy, valuetable, env, gamma=0.9, theta=1e-10):
    while True:
        delta = 0
        for state in range(env.observation_space.n):
            v = 0
            action = policy[state]
            for prob, nextstate, reward, done in env.P[state][action]:
                v += prob * (reward + gamma * valuetable[nextstate])
            delta = max(delta, abs(valuetable[state] - v))
            valuetable[state] = v
        if delta < theta:
            break
    return value_table
def policyimprovement(valuetable, policy, env, gamma=0.9):
    policystable = True
    for state in range(env.observation_space.n):
        oldaction = policy[state]
        actionvalues = np.zeros(env.action_space.n)
        for action in range(env.action_space.n):
            for prob, nextstate, reward, done in env.P[state][action]:
                actionvalues[action] += prob * (reward + gamma * valuetable[nextstate])
        policy[state] = np.argmax(actionvalues)
        if oldaction != policy[state]:
            policystable = False
    return policy, policystable

def policyiteration(env, gamma=0.9, theta=1e-10):
    policy = np.random.choice(env.action_space.n, size=env.observation_space.n)
    valuetable = np.zeros(env.observation_space.n)
    while True:
        valuetable = policy_evaluation(policy, value_table, env, gamma, theta)
        policy, policystable = policyimprovement(value_table, policy, env, gamma)
        if policystable:
            return policy, value_table
optimalpolicy, optimalvalue = policyiteration(env, gamma, theta)
print("\nOptimal Policy:")
print(optimalpolicy)
print("\nOptimal Value Function:")
print(optimalvalue.reshape(4,4))

Number of states: 16
Number of actions: 4

Optimal Policy:
[1 2 1 0 1 0 1 0 2 1 1 0 0 2 2 0]

Optimal Value Function:
[[0.59049 0.6561  0.729   0.6561 ]
 [0.6561  0.      0.81    0.     ]
 [0.729   0.81    0.9     0.     ]
 [0.      0.9     1.      0.     ]]
